# Leetcode < id > 
## < Leetcode Problem's name >

In [0]:
# https://leetcode.com/problems/find-total-time-spent-by-each-employee
# Completed at 2026/04/17

In [0]:
# Frameworks
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
import pyspark.sql.functions as F
from datetime import datetime

## Part 1: Parsing the raw data into PySpark Dataframes

In [0]:
# Dataframe initialization function
def parse_raw_to_df(table_str):
    spark = SparkSession.builder.getOrCreate()
    lines = []

    # Step 1 — filter lines
    for line in table_str.splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith("+"):
            continue

        lines.append(line)

    # Step 2 — extract schema
    schema_cols = []
    schema_types = []
    i = 1  # skip "| Column Name | Type |"

    while "Column Name" not in lines[0]:
        i += 1

    # Actually schema rows start after header row
    i = 1

    while "|" in lines[i] and "emp_id" in table_str:
        parts = [p.strip() for p in lines[i].split("|") if p.strip()]

        if len(parts) == 2:
            schema_cols.append(parts[0])
            schema_types.append(parts[1])

        else:
            break

        i += 1

        # Stop when next section starts
        if i >= len(lines):
            break

        if len(parts) != 2:
            break

    # Step 3 — map Spark types
    type_map = {
        "int": IntegerType(),
        "string": StringType(),
        "date": DateType()
    }

    fields = []

    for col, typ in zip(schema_cols, schema_types):
        spark_type = type_map.get(typ.lower(), StringType())
        fields.append(
            StructField(col, spark_type, True)
        )

    schema = StructType(fields)

    # Step 4 — extract data rows
    data_rows = []

    # find start of real data
    data_start = None

    for idx, line in enumerate(lines):
        if schema_cols[0] in line:
            data_start = idx + 1

    # parse rows
    for line in lines[data_start:]:
        parts = [p.strip() for p in line.split("|") if p.strip()]

        if len(parts) != len(schema_cols):
            continue

        converted_row = []

        for val, typ in zip(parts, schema_types):

            if typ == "int":
                converted_row.append(int(val))

            elif typ == "date":
                converted_row.append(
                    datetime.strptime(val, "%Y-%m-%d").date()
                )

            else:
                converted_row.append(val)

        data_rows.append(tuple(converted_row))

    # Step 5 — create dataframe
    df = spark.createDataFrame(data_rows, schema)

    return df

In [0]:
# Initial data tables

# EMPLOYEE
r_employees = """
+-------------+------+
| Column Name | Type |
+-------------+------+
| emp_id      | int  |
| event_day   | date |
| in_time     | int  |
| out_time    | int  |
+-------------+------+
+--------+------------+---------+----------+
| emp_id | event_day  | in_time | out_time |
+--------+------------+---------+----------+
| 1      | 2020-11-28 | 4       | 32       |
| 1      | 2020-11-28 | 55      | 200      |
| 1      | 2020-12-03 | 1       | 42       |
| 2      | 2020-11-28 | 3       | 33       |
| 2      | 2020-12-09 | 47      | 74       |
+--------+------------+---------+----------+
"""
employees_df = parse_raw_to_df(r_employees)
employees_df.createOrReplaceTempView("employees")

In [0]:
employees_df.printSchema()

root
 |-- emp_id: integer (nullable = true)
 |-- event_day: date (nullable = true)
 |-- in_time: integer (nullable = true)
 |-- out_time: integer (nullable = true)
 |-- total_time: integer (nullable = true)



In [0]:
employees_df.display(5)

emp_id,event_day,in_time,out_time
1,2020-11-28,4,32
1,2020-11-28,55,200
1,2020-12-03,1,42
2,2020-11-28,3,33
2,2020-12-09,47,74


## Part 2: Querying and Solution

Write a solution to calculate the total time in minutes spent by each employee on each day at the office. Note that within one day, an employee can enter and leave more than once. The time spent in the office for a single entry is out_time - in_time.

Return the result table in any order.

![image_1776473371498.png](./image_1776473371498.png "image_1776473371498.png")

### SparkSQL

In [0]:
sol_sql = spark.sql("""

                    """).show()

+----------+------+----------+
|       day|emp_id|total_time|
+----------+------+----------+
|2020-11-28|     1|       173|
|2020-12-03|     1|        41|
|2020-11-28|     2|        30|
|2020-12-09|     2|        27|
+----------+------+----------+



In [0]:
%sql
SELECT
FROM

day,emp_id,total_time
2020-11-28,1,173
2020-12-03,1,41
2020-11-28,2,30
2020-12-09,2,27


### PySpark


In [0]:
# Pyspark code

+----------+------+---------------+
| event_day|emp_id|sum(total_time)|
+----------+------+---------------+
|2020-11-28|     1|            173|
|2020-12-03|     1|             41|
|2020-11-28|     2|             30|
|2020-12-09|     2|             27|
+----------+------+---------------+

